# snobal-box quickstart

Run the snobal point snow model on the bundled Atwater example, or on your
own hourly forcing. Works in Colab or locally.

In [ ]:
# Setup. In Colab this installs the package; locally it uses what is installed.
import importlib.util, subprocess, sys
REPO = 'https://github.com/MacSkiles/snobal-in-a-box'
if importlib.util.find_spec('snobal_box') is None:
    subprocess.run(['git', 'clone', '-q', REPO, 'snobal-in-a-box'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', './snobal-in-a-box'], check=True)
    %cd snobal-in-a-box
import snobal_box as sb, pandas as pd
print('snobal-box', sb.__version__)

## A. The example

In situ station forcing at Atwater Study Plot, WY2022, with three warming
scenarios that separate the energy-balance response from the phase response.

In [ ]:
res = sb.run_config('examples/atwater_wy22_insitu.yaml')
res.summary[['swe_peak', 'swe_peak_date', 'swe_half_melt_date', 'swe_melt_out_date',
             'snowfall_mm', 'rain_on_snow_mm', 'snow_fraction', 'closure_residual_mm']]

In [ ]:
from snobal_box import plots
plots.plot_swe_depth(res.outputs, res.config['name']);

## B. Your own forcing

1. Make an hourly CSV with columns `time, S_n, I_lw, T_a, e_a, u, T_g, m_pp`
   (units and meanings in `snobal_box/templates/my_site.yaml`; temperatures in C).
2. Upload it (Colab: the folder icon on the left, or run the next cell).
3. Edit the config below and run.

In [ ]:
try:
    from google.colab import files
    uploaded = list(files.upload())
    MY_CSV = uploaded[0]
except ImportError:
    MY_CSV = 'snobal_box/templates/forcing_template.csv'   # local: point at your file
MY_CSV

In [ ]:
config = {
    'name': 'My site',
    'site': {
        'elevation_m': 2670,
        'heights': {'z_u': 5.0, 'z_t': 5.0, 'z_0': 0.003, 'z_g': 0.15},
        'initial_snow': {'depth_m': 0.0, 'density': 300, 'temp_C': -4.0},
    },
    'forcing': {'format': 'csv', 'file': MY_CSV, 'timezone': 'UTC'},
    'precip': {'phase': 'table', 'density': 'table'},
    'scenarios': {'+2C': {'T_offset': 2.0},
                  '+2C energy only': {'T_offset': 2.0, 'rebuild_phase': False}},
    'output': {'dir': 'output'},
}
mine = sb.run_config(config)
mine.summary

In [ ]:
plots.plot_swe_depth(mine.outputs, config['name']);